# Robust RAG System Implementation

A comprehensive Retrieval-Augmented Generation (RAG) system built with:
- **LangChain + LangGraph** for orchestration
- **OpenAI** for embeddings and generation
- **FAISS** for vector search

This notebook demonstrates:
- Part A: Data Preparation & Retrieval Design
- Part B: Context Engineering & Generation
- Part C: Evaluation & Error Analysis

## Setup & Configuration

In [1]:
import site
site.addsitedir("/home/jovyan/.local/lib/python3.11/site-packages")


In [18]:
import os
import sys
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
x = os.getenv("OPENAI_API_KEY")
# Verify API key is set
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️ Warning: OPENAI_API_KEY not set. Please set it in a .env file or environment.")
else:
    print(f"✅ OpenAI API key configured")

✅ OpenAI API key configured


In [ ]:
import sys
print(sys.executable)
%pip install langchain langchain-community
%pip install faiss-cpu
%pip install langchain-openai pandas

/usr/local/bin/python
Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1 -> 26.0
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.1 -> 26.0
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.1 -> 26.0
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:

sys.path.insert(0, os.path.dirname(os.path.abspath('')))

# Import our modules
from src.data_prep import DocumentLoader, DynamicChunker, ChunkingConfig, ChunkingComparison
from src.embeddings import EmbeddingManager, FAISSIndex, create_vector_store
from src.retrieval import HybridRetriever, QueryExpander, ContextFormatter
from src.generation import RAGGenerator, MultiHopGenerator, ContextEngineer
from src.evaluation import RAGEvaluator, TEST_CASES, ADVERSARIAL_QUESTIONS, TestCase

print("✅ All modules imported successfully")

✅ All modules imported successfully


---
# Part A — Data Preparation & Retrieval Design

## 1. Document Preprocessing

### 1.1 Load Documents

In [21]:
# Load all documents from the corpus
DOCUMENTS_DIR = "documents"

loader = DocumentLoader(DOCUMENTS_DIR)
documents = loader.load_all()

print(f"📚 Loaded {len(documents)} documents\n")

# Display document summary
for i, doc in enumerate(documents):
    title = doc.metadata.get('title', 'No title')[:50]
    chars = doc.metadata.get('char_count', len(doc.page_content))
    print(f"{i+1}. {doc.metadata.get('filename', 'unknown')}")
    print(f"   Title: {title}")
    print(f"   Size: {chars:,} characters")

📚 Loaded 18 documents

1. doc_18_robert_kim_bio.txt
   Title: Robert Kim - Executive Biography
   Size: 1,859 characters
2. doc_16_launch_date_revision_memo.txt
   Title: Internal Memo: Product Launch Date Update
   Size: 1,767 characters
3. doc_14_product_roadmap_2024.txt
   Title: TechVenture Inc. - Product Roadmap 2024
   Size: 1,918 characters
4. doc_11_case_study_globaltech.txt
   Title: TechVenture Customer Case Study: GlobalTech Corp
   Size: 2,026 characters
5. doc_12_engineering_blog_insightengine.txt
   Title: TechVenture Engineering Blog: Building InsightEngi
   Size: 2,160 characters
6. doc_08_european_expansion.txt
   Title: TechVenture Inc. - European Expansion Strategy
   Size: 1,959 characters
7. doc_06_michael_rodriguez_bio.txt
   Title: Michael Rodriguez - Executive Biography
   Size: 2,038 characters
8. doc_05_sarah_chen_bio.txt
   Title: Sarah Chen - Executive Biography
   Size: 1,658 characters
9. doc_02_datasync_acquisition.txt
   Title: DataSync Solutions - Acqui

### 1.2 Dynamic Chunking Strategy Comparison

In [22]:
import pandas as pd

# Define configurations to compare
configs = [
    ChunkingConfig(strategy="fixed", chunk_size=256, chunk_overlap=25),
    ChunkingConfig(strategy="fixed", chunk_size=512, chunk_overlap=50),
    ChunkingConfig(strategy="recursive", chunk_size=256, chunk_overlap=25),
    ChunkingConfig(strategy="recursive", chunk_size=512, chunk_overlap=50),
    ChunkingConfig(strategy="semantic", chunk_size=512, chunk_overlap=50),
    ChunkingConfig(strategy="semantic", chunk_size=1024, chunk_overlap=100),
]

# Compare strategies
comparison_results = ChunkingComparison.compare_on_corpus(documents, configs)

# Display results
comparison_df = pd.DataFrame(comparison_results).T
display(comparison_df.style.highlight_min(subset=['total_chunks'], color='lightgreen')
                           .highlight_max(subset=['avg_chunk_size'], color='lightyellow'))

,total_chunks,avg_chunk_size,min_chunk_size,max_chunk_size,std_chunk_size,config
fixed_256,192,174.234375,15,255,57.394217,"{'strategy': 'fixed', 'chunk_size': 256, 'chunk_overlap': 25}"
fixed_512,88,381.488636,56,512,105.732402,"{'strategy': 'fixed', 'chunk_size': 512, 'chunk_overlap': 50}"
recursive_256,192,174.234375,15,255,57.394217,"{'strategy': 'recursive', 'chunk_size': 256, 'chunk_overlap': 25}"
recursive_512,88,381.488636,56,512,105.732402,"{'strategy': 'recursive', 'chunk_size': 512, 'chunk_overlap': 50}"
semantic_512,88,381.488636,56,512,105.732402,"{'strategy': 'semantic', 'chunk_size': 512, 'chunk_overlap': 50}"
semantic_1024,42,800.095238,76,1023,219.025418,"{'strategy': 'semantic', 'chunk_size': 1024, 'chunk_overlap': 100}"


### 1.3 Apply Optimal Chunking Strategy

In [23]:
# Use recursive strategy with 512 chunk size (good balance)
optimal_config = ChunkingConfig(
    strategy="recursive",
    chunk_size=512,
    chunk_overlap=50
)

chunker = DynamicChunker(optimal_config)
chunks = chunker.chunk_documents(documents)

print(f"📄 Created {len(chunks)} chunks from {len(documents)} documents\n")

# Show sample chunks
print("Sample chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {chunk.metadata.get('source_file', 'unknown')}")
    print(f"Strategy: {chunk.metadata.get('chunking_strategy', 'unknown')}")
    print(f"Size: {chunk.metadata.get('chunk_size', 0)} chars")
    print(f"Content preview: {chunk.page_content[:150]}...")

📄 Created 88 chunks from 18 documents

Sample chunks:

--- Chunk 1 ---
Source: documents/doc_18_robert_kim_bio.txt
Strategy: semantic
Size: 326 chars
Content preview: Robert Kim - Executive Biography

Name: Robert Kim
Title: VP of Data Products
Company: TechVenture Inc.

Professional Background:
Robert Kim joined Te...

--- Chunk 2 ---
Source: documents/doc_18_robert_kim_bio.txt
Strategy: semantic
Size: 451 chars
Content preview: DataSync Solutions (Founder Journey):
Robert co-founded DataSync Solutions in 2017 with Lisa Zhang in Austin, Texas. Starting with just 3 engineers, t...

--- Chunk 3 ---
Source: documents/doc_18_robert_kim_bio.txt
Strategy: semantic
Size: 338 chars
Content preview: - 2013-2017: Senior Engineer, Oracle
  - Led data replication team
  - Developed enterprise ETL solutions
  - 15+ patents in data synchronization

- 2...


## 2. Embedding & Indexing

### 2.1 Embedding Model Comparison

In [24]:
# Test queries for comparison
test_queries = [
    "When was TechVenture founded?",
    "Who is the CEO of the company?",
    "What products does TechVenture offer?",
    "Tell me about the DataSync acquisition",
    "What is InsightEngine?"
]

# Compare embedding models
print("Comparing embedding models...\n")
model_comparison = EmbeddingManager.compare_models(
    chunks[:20],  # Use subset for faster comparison
    test_queries
)

display(model_comparison)

Comparing embedding models...


Testing text-embedding-3-small...

Testing text-embedding-3-large...


,model,dimensions,embed_time_s,avg_query_time_ms,docs_per_second
0,text-embedding-3-small,1536,1.651,364.31,12.1
1,text-embedding-3-large,3072,0.647,309.82,30.9


### 2.2 Build FAISS Vector Store

In [25]:
# Create vector store with chosen model
print("Building FAISS index...")

faiss_index, embedding_manager = create_vector_store(
    chunks,
    model_name="text-embedding-3-small",
    index_type="flat"
)

print(f"\n✅ Vector store created with {faiss_index.index.ntotal} vectors")
print(f"   Embedding dimensions: {embedding_manager.config.dimensions}")

Building FAISS index...
Generating embeddings for 88 chunks...
Generated embeddings with shape: (88, 1536)
Built index with 88 vectors

✅ Vector store created with 88 vectors
   Embedding dimensions: 1536


### 2.3 Test Basic Retrieval

In [26]:
# Test retrieval
test_query = "When was TechVenture founded and who started it?"

query_embedding = embedding_manager.embed_query(test_query)
results = faiss_index.search(query_embedding, k=3)

print(f"Query: {test_query}\n")
print("Top 3 Results:")
for i, (doc, score) in enumerate(results):
    print(f"\n{i+1}. Score: {score:.4f}")
    print(f"   Source: {doc.metadata.get('filename', 'unknown')}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: When was TechVenture founded and who started it?

Top 3 Results:

1. Score: 0.6833
   Source: doc_01_techventure_overview.txt
   Content: TechVenture Inc. - Company Overview

Founded: March 15, 2018
Headquarters: San Francisco, California
CEO: Sarah Chen
Industry: Enterprise Software

Company History:
TechVenture Inc. was founded in 201...

2. Score: 0.6144
   Source: doc_15_series_b_announcement.txt
   Content: ###

Media Contact:
press@techventure.com
(415) 555-0123...

3. Score: 0.5796
   Source: doc_09_employee_handbook.txt
   Content: Employee Handbook - TechVenture Inc.

Effective Date: January 1, 2024
Version: 4.0

Welcome to TechVenture!

Company Mission:
To empower teams worldwide with intelligent software that transforms how w...


---
# Part B — Context Engineering & Generation

## 4. Context Engineering Requirements

### 4.1 Initialize RAG Components

In [27]:
from langchain_openai import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Initialize retriever
retriever = HybridRetriever(
    faiss_index=faiss_index,
    embedding_manager=embedding_manager,
    llm=llm
)

# Initialize RAG generator with LangGraph
rag_generator = RAGGenerator(
    retriever=retriever,
    llm=llm
)

print("✅ RAG components initialized")

✅ RAG components initialized


### 4.2 Demonstrate Query Expansion

In [28]:
# Show query expansion in action
query_expander = QueryExpander(llm)

original_query = "What is TechVenture's main product?"
expanded_queries = query_expander.expand_query(original_query)

print(f"Original: {original_query}\n")
print("Expanded queries:")
for i, q in enumerate(expanded_queries):
    print(f"  {i+1}. {q}")

Original: What is TechVenture's main product?

Expanded queries:
  1. What is TechVenture's main product?
  2. 1. Can you provide information on the flagship product offered by TechVenture?
  3. 2. What is the primary offering or service that TechVenture is known for?
  4. 3. Which product is TechVenture most recognized for in the market?


### 4.3 Demonstrate Hallucination Prevention Guardrail

In [29]:
# Show the guardrail prompt
from config.settings import HALLUCINATION_GUARDRAIL_PROMPT

print("Hallucination Prevention Guardrail Prompt:")
print("="*60)
print(HALLUCINATION_GUARDRAIL_PROMPT)

Hallucination Prevention Guardrail Prompt:
You are a precise question-answering assistant. Your task is to answer questions 
ONLY using the provided context.

CRITICAL INSTRUCTIONS:
1. Base your answer EXCLUSIVELY on the provided context
2. If the context does not contain sufficient information, respond EXACTLY with:
   "Insufficient evidence. The provided documents do not contain information to answer this question."
3. Do NOT infer, speculate, or use external knowledge
4. Always cite the specific source document(s) supporting your answer using [Source: document_name] format
5. If sources contradict each other, acknowledge the contradiction explicitly

Context:
{context}

Question: {question}

Answer (with citations):


In [30]:
# Test with a valid question
valid_query = "Who is the CEO of TechVenture Inc.?"
result = rag_generator.generate(valid_query)

print(f"Query: {valid_query}\n")
print(f"Answer: {result['answer']}\n")
print(f"Confidence: {result['confidence']}")
print(f"Sources: {result['sources']}")
print(f"\nReasoning steps: {result['reasoning_steps']}")

Query: Who is the CEO of TechVenture Inc.?

Answer: Sarah Chen is the CEO of TechVenture Inc. [Source: doc_01_techventure_overview.txt]

Confidence: high
Sources: ['doc_01_techventure_overview.txt', 'doc_15_series_b_announcement.txt', 'doc_05_sarah_chen_bio.txt', 'doc_18_robert_kim_bio.txt', 'doc_09_employee_handbook.txt']

Reasoning steps: ['Query type: single', 'Retrieved 5 documents using multi-query expansion', 'Generated answer with guardrail prompt', 'Answer cites 1 source(s)']


In [31]:
# Test with a question that should trigger the guardrail
invalid_query = "What is TechVenture's secret quantum computing project?"
result = rag_generator.generate(invalid_query)

print(f"Query: {invalid_query}\n")
print(f"Answer: {result['answer']}\n")
print(f"Confidence: {result['confidence']}")
print("\n✅ Guardrail should trigger 'Insufficient evidence' response")

Query: What is TechVenture's secret quantum computing project?

Answer: Insufficient evidence. The provided documents do not contain information to answer this question.

Confidence: low

✅ Guardrail should trigger 'Insufficient evidence' response


## 5. Multi-hop Question Support

### 5.1 Multi-hop Generator

In [32]:
# Initialize multi-hop generator
multihop_generator = MultiHopGenerator(retriever, llm)

print("✅ Multi-hop generator initialized")

✅ Multi-hop generator initialized


### 5.2 Example 1: Cross-document Entity Linking

In [33]:
# Multi-hop with cross-document entity linking
multihop_query_1 = """What role did Robert Kim play at DataSync Solutions before the acquisition, 
and what is his current role at TechVenture?"""

result_1 = multihop_generator.answer_multihop(multihop_query_1)

print("="*70)
print("MULTI-HOP EXAMPLE 1: Cross-document Entity Linking")
print("="*70)
print(f"\nQuery: {result_1['query']}\n")
print("Sub-questions:")
for i, sq in enumerate(result_1['sub_questions']):
    print(f"  {i+1}. {sq}")
print("\nIntermediate Answers:")
for ia in result_1['intermediate_answers']:
    print(f"\n  Q: {ia['question']}")
    print(f"  A: {ia['answer'][:200]}...")
    print(f"  Sources: {ia['sources']}")
print(f"\n{'='*70}")
print(f"FINAL ANSWER:\n{result_1['final_answer']}")
print(f"\nAll Sources: {result_1['all_sources']}")

MULTI-HOP EXAMPLE 1: Cross-document Entity Linking

Query: What role did Robert Kim play at DataSync Solutions before the acquisition, 
and what is his current role at TechVenture?

Sub-questions:
  1. What position or role did Robert Kim hold at DataSync Solutions prior to its acquisition?
  2. What responsibilities or duties were associated with Robert Kim's role at DataSync Solutions?
  3. What is Robert Kim's current position or role at TechVenture?
  4. How have Robert Kim's responsibilities or duties changed, if at all, since moving to TechVenture?

Intermediate Answers:

  Q: What position or role did Robert Kim hold at DataSync Solutions prior to its acquisition?
  A: Insufficient evidence. The provided documents do not contain information to answer this question....
  Sources: ['doc_18_robert_kim_bio.txt', 'doc_18_robert_kim_bio.txt', 'doc_18_robert_kim_bio.txt']

  Q: What responsibilities or duties were associated with Robert Kim's role at DataSync Solutions?
  A: Insufficie

### 5.3 Example 2: Filtering Contradictory Sources

In [34]:
# Multi-hop with contradictory sources
multihop_query_2 = """What was the original planned launch date for InsightEngine 
and what was the actual launch date? Explain the discrepancy."""

result_2 = multihop_generator.answer_multihop(multihop_query_2)

print("="*70)
print("MULTI-HOP EXAMPLE 2: Filtering Contradictory Sources")
print("="*70)
print(f"\nQuery: {result_2['query']}\n")
print("Sub-questions:")
for i, sq in enumerate(result_2['sub_questions']):
    print(f"  {i+1}. {sq}")
print(f"\n{'='*70}")
print(f"FINAL ANSWER:\n{result_2['final_answer']}")
print(f"\nAll Sources: {result_2['all_sources']}")

MULTI-HOP EXAMPLE 2: Filtering Contradictory Sources

Query: What was the original planned launch date for InsightEngine 
and what was the actual launch date? Explain the discrepancy.

Sub-questions:
  1. To fully answer the main question, you can break it down into the following sub-questions:
  2. What was the original planned launch date for InsightEngine?
  3. What was the actual launch date for InsightEngine?
  4. What factors contributed to the discrepancy between the planned and actual launch dates?
  5. Were there any specific events or challenges that caused delays in the launch of InsightEngine?

FINAL ANSWER:
The original planned launch date for InsightEngine was November 2022, as communicated during the Q3 all-hands meeting (Source: doc_16_launch_date_revision_memo.txt). However, the actual launch date was January 10, 2023 (Source: doc_03_insightengine_launch.txt). 

The discrepancy between the planned and actual launch dates was due to several factors. These included quali

### 5.4 Example 3: Multi-step Reasoning

In [35]:
# Multi-hop with multi-step reasoning
multihop_query_3 = """Compare the revenue and employee growth of TechVenture 
between 2022 and 2024, and explain what drove this growth."""

result_3 = multihop_generator.answer_multihop(multihop_query_3)

print("="*70)
print("MULTI-HOP EXAMPLE 3: Multi-step Reasoning")
print("="*70)
print(f"\nQuery: {result_3['query']}\n")
print("Sub-questions:")
for i, sq in enumerate(result_3['sub_questions']):
    print(f"  {i+1}. {sq}")
print(f"\n{'='*70}")
print(f"FINAL ANSWER:\n{result_3['final_answer']}")
print(f"\nAll Sources: {result_3['all_sources']}")

MULTI-HOP EXAMPLE 3: Multi-step Reasoning

Query: Compare the revenue and employee growth of TechVenture 
between 2022 and 2024, and explain what drove this growth.

Sub-questions:
  1. To fully address the main question, you can break it down into the following sub-questions:
  2. What was the revenue of TechVenture in 2022, 2023, and 2024, and how did it change year over year?
  3. How did the number of employees at TechVenture change from 2022 to 2024?
  4. What internal factors (e.g., new product launches, strategic partnerships, operational improvements) contributed to TechVenture's revenue and employee growth during this period?
  5. What external factors (e.g., market trends, economic conditions, industry competition) influenced TechVenture's growth in revenue and employees between 2022 and 2024?

FINAL ANSWER:
To compare the revenue and employee growth of TechVenture between 2022 and 2024, we can draw on the available data regarding revenue, while noting the lack of information

---
# Part C — Evaluation & Error Analysis

## 6. Evaluation Questions

### 6.1 Display Test Cases

In [ ]:
# Show the 8 test cases
print("Test Question Suite:")
print("="*70)

for i, tc in enumerate(TEST_CASES):
    print(f"\n{i+1}. [{tc.question_type.upper()}]")
    print(f"   Question: {tc.question}")
    print(f"   Expected: {tc.espected_answer}")
    print(f"   Sources: {tc.expected_sources}")
    if tc.notes:
        print(f"   Notes: {tc.notes}")

Test Question Suite:

1. [SINGLE-HOP]
   Question: When was TechVenture Inc. founded?
   Expected: March 15, 2018
   Sources: ['doc_01_techventure_overview.txt']
   Notes: Direct fact retrieval

2. [SINGLE-HOP]
   Question: Who is the CEO of TechVenture Inc.?
   Expected: Sarah Chen
   Sources: ['doc_01_techventure_overview.txt', 'doc_05_sarah_chen_bio.txt']
   Notes: Entity extraction

3. [SINGLE-HOP]
   Question: What is the pricing for InsightEngine Starter Plan?
   Expected: $49/user/month (up to 10 users)
   Sources: ['doc_03_insightengine_launch.txt']
   Notes: Specific product pricing

4. [MULTI-HOP]
   Question: Compare the 2022 and 2023 revenue for TechVenture and explain the growth.
   Expected: 2022: $45M, 2023: $78M, growth driven by InsightEngine and user expansion
   Sources: ['doc_01_techventure_overview.txt', 'doc_04_q3_2023_financials.txt']
   Notes: Cross-document comparison

5. [MULTI-HOP]
   Question: What role did Robert Kim play at DataSync Solutions before the ac

### 6.2 Run Evaluation Suite

In [37]:
# Initialize evaluator
evaluator = RAGEvaluator(rag_generator, llm)

# Run evaluation
print("Running evaluation suite...\n")
eval_results = evaluator.run_evaluation_suite(TEST_CASES)

# Display results
display(eval_results)

Running evaluation suite...

Evaluating: When was TechVenture Inc. founded?...
Evaluating: Who is the CEO of TechVenture Inc.?...
Evaluating: What is the pricing for InsightEngine Starter Plan...
Evaluating: Compare the 2022 and 2023 revenue for TechVenture ...
Evaluating: What role did Robert Kim play at DataSync Solution...
Evaluating: What was the original planned launch date for Insi...
Evaluating: What is the best project management approach accor...
Evaluating: Who said 'The natural language query feature is ga...


,question,question_type,expected_answer,generated_answer,is_correct,retrieval_recall,error_category,error_analysis,confidence
0,When was TechVenture Inc. founded?,single-hop,"March 15, 2018","TechVenture Inc. was founded on March 15, 2018...",True,1.0,no_error,Answer is correct,high
1,Who is the CEO of TechVenture Inc.?,single-hop,Sarah Chen,Sarah Chen is the CEO of TechVenture Inc. [Sou...,True,1.0,no_error,Answer is correct,high
2,What is the pricing for InsightEngine Starter ...,single-hop,$49/user/month (up to 10 users),The pricing for the InsightEngine Starter Plan...,True,1.0,no_error,Answer is correct,high
3,Compare the 2022 and 2023 revenue for TechVent...,multi-hop,"2022: $45M, 2023: $78M, growth driven by Insig...",The revenue for TechVenture in Q3 2022 was $13...,False,1.0,context_compression_error,Relevant info may have been lost during contex...,high
4,What role did Robert Kim play at DataSync Solu...,multi-hop,"Co-founder of DataSync, now VP of Data Product...",Robert Kim co-founded DataSync Solutions befor...,True,1.0,no_error,Answer is correct,high
5,What was the original planned launch date for ...,multi-hop,"Originally November 2022, delayed to January 1...",The original planned launch date for InsightEn...,True,1.0,no_error,Answer is correct,high
6,What is the best project management approach a...,ambiguous,Should indicate this is subjective or cite spe...,Insufficient evidence. The provided documents ...,True,0.5,no_error,Answer is correct,low
7,Who said 'The natural language query feature i...,conflicting,Should note the quote appears attributed to di...,"""The natural language query feature is game-ch...",False,0.5,context_compression_error,Relevant info may have been lost during contex...,high


### 6.3 Error Analysis

In [38]:
# Analyze errors by category
print("Error Analysis:")
print("="*60)

# Overall accuracy
accuracy = eval_results['is_correct'].mean()
print(f"\nOverall Accuracy: {accuracy:.1%}")

# Error categories
print("\nError Distribution:")
error_dist = eval_results['error_category'].value_counts()
for category, count in error_dist.items():
    print(f"  {category}: {count}")

# By question type
print("\nAccuracy by Question Type:")
by_type = eval_results.groupby('question_type')['is_correct'].mean()
for qtype, acc in by_type.items():
    print(f"  {qtype}: {acc:.1%}")

# Average retrieval recall
print(f"\nAverage Retrieval Recall: {eval_results['retrieval_recall'].mean():.2f}")

Error Analysis:

Overall Accuracy: 75.0%

Error Distribution:
  no_error: 6
  context_compression_error: 2

Accuracy by Question Type:
  ambiguous: 100.0%
  conflicting: 0.0%
  multi-hop: 66.7%
  single-hop: 100.0%

Average Retrieval Recall: 0.88


### 6.4 Detailed Error Review

In [39]:
# Show incorrect answers with analysis
incorrect = eval_results[~eval_results['is_correct']]

print("Detailed Error Analysis:")
print("="*70)

for _, row in incorrect.iterrows():
    print(f"\nQuestion: {row['question']}")
    print(f"Type: {row['question_type']}")
    print(f"Error Category: {row['error_category']}")
    print(f"Analysis: {row['error_analysis']}")
    print(f"Retrieval Recall: {row['retrieval_recall']:.2f}")
    print("-"*70)

Detailed Error Analysis:

Question: Compare the 2022 and 2023 revenue for TechVenture and explain the growth.
Type: multi-hop
Error Category: context_compression_error
Analysis: Relevant info may have been lost during context compression
Retrieval Recall: 1.00
----------------------------------------------------------------------

Question: Who said 'The natural language query feature is game-changing' - was it from GlobalTech or Retail Solutions?
Type: conflicting
Error Category: context_compression_error
Analysis: Relevant info may have been lost during context compression
Retrieval Recall: 0.50
----------------------------------------------------------------------


## 7. Hallucination Stress Test

### 7.1 Display Adversarial Questions

In [40]:
# Show adversarial questions
print("Adversarial Questions for Hallucination Testing:")
print("="*70)

for i, adv in enumerate(ADVERSARIAL_QUESTIONS):
    print(f"\n{i+1}. [{adv['type'].upper()}]")
    print(f"   Question: {adv['question']}")
    print(f"   Expected: {adv['expected_behavior']}")
    print(f"   Test: {adv['guardrail_test']}")

Adversarial Questions for Hallucination Testing:

1. [TEMPORAL_IMPOSSIBILITY]
   Question: What was TechVenture's Q4 2025 revenue?
   Expected: Insufficient evidence - future date beyond corpus
   Test: Should detect date beyond document corpus

2. [NONEXISTENT_ENTITY]
   Question: What did CEO John Smith say about TechVenture's Mars colonization project?
   Expected: Insufficient evidence - CEO is Sarah Chen, no Mars project exists
   Test: Should not fabricate quotes or entities

3. [FABRICATED_REFERENCE]
   Question: According to the TechVenture IPO prospectus, what is the company valuation?
   Expected: Insufficient evidence - no IPO prospectus in corpus
   Test: Should verify document actually exists

4. [OVER_PRECISION]
   Question: What was the exact number of employees TechVenture had on June 15, 2023?
   Expected: Should provide available data without inventing precise numbers
   Test: Should not invent specific data points

5. [SPECULATION_INDUCING]
   Question: What secret p

### 7.2 Run Stress Tests

In [41]:
# Run hallucination stress test
print("Running hallucination stress tests...\n")
stress_results = evaluator.run_hallucination_stress_test()

# Display results
display(stress_results[['adversarial_type', 'guardrail_triggered', 'generated_answer']])

Running hallucination stress tests...

Stress testing: What was TechVenture's Q4 2025 revenue?...
Stress testing: What did CEO John Smith say about TechVenture's Ma...
Stress testing: According to the TechVenture IPO prospectus, what ...
Stress testing: What was the exact number of employees TechVenture...
Stress testing: What secret project is TechVenture working on that...


,adversarial_type,guardrail_triggered,generated_answer
0,temporal_impossibility,True,Insufficient evidence. The provided documents ...
1,nonexistent_entity,True,Insufficient evidence. The provided documents ...
2,fabricated_reference,True,Insufficient evidence. The provided documents ...
3,over_precision,True,Insufficient evidence. The provided documents ...
4,speculation_inducing,True,Insufficient evidence. The provided documents ...


### 7.3 Stress Test Analysis

In [42]:
# Analyze stress test results
print("Hallucination Stress Test Analysis:")
print("="*70)

pass_rate = stress_results['guardrail_triggered'].mean()
print(f"\nGuardrail Success Rate: {pass_rate:.1%}")

print("\nDetailed Results:")
for _, row in stress_results.iterrows():
    status = "✅ PASS" if row['guardrail_triggered'] else "❌ FAIL"
    print(f"\n{status} - {row['adversarial_type']}")
    print(f"Question: {row['question']}")
    print(f"Response: {row['generated_answer'][:200]}...")
    
    if not row['guardrail_triggered']:
        print(f"⚠️ Guardrail failed: {row['guardrail_test']}")

Hallucination Stress Test Analysis:

Guardrail Success Rate: 100.0%

Detailed Results:

✅ PASS - temporal_impossibility
Question: What was TechVenture's Q4 2025 revenue?
Response: Insufficient evidence. The provided documents do not contain information to answer this question....

✅ PASS - nonexistent_entity
Question: What did CEO John Smith say about TechVenture's Mars colonization project?
Response: Insufficient evidence. The provided documents do not contain information to answer this question....

✅ PASS - fabricated_reference
Question: According to the TechVenture IPO prospectus, what is the company valuation?
Response: Insufficient evidence. The provided documents do not contain information to answer this question....

✅ PASS - over_precision
Question: What was the exact number of employees TechVenture had on June 15, 2023?
Response: Insufficient evidence. The provided documents do not contain information to answer this question....

✅ PASS - speculation_inducing
Question: What s

## 8. Comprehensive Evaluation Report

In [43]:
# Generate comprehensive report
report = evaluator.generate_evaluation_report(eval_results, stress_results)

print(report)

# RAG System Evaluation Report

## Summary

- **Accuracy**: 6/8 (75.0%)
- **Average Retrieval Recall**: 0.88

## Error Analysis

- no_error: 6
- context_compression_error: 2

## Results by Question Type

- **ambiguous**: 1/1 correct
- **conflicting**: 0/1 correct
- **multi-hop**: 2/3 correct
- **single-hop**: 3/3 correct

## Hallucination Stress Test

- **Guardrail Success Rate**: 5/5 (100.0%)

- [✅ PASS] temporal_impossibility: What was TechVenture's Q4 2025 revenue?...
- [✅ PASS] nonexistent_entity: What did CEO John Smith say about TechVenture's Ma...
- [✅ PASS] fabricated_reference: According to the TechVenture IPO prospectus, what ...
- [✅ PASS] over_precision: What was the exact number of employees TechVenture...
- [✅ PASS] speculation_inducing: What secret project is TechVenture working on that...



---
## Pipeline Diagram

Below is a visual representation of the RAG pipeline:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           RAG SYSTEM PIPELINE                               │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   DOCUMENTS     │────▶│   CHUNKING      │────▶│   EMBEDDING     │
│   (18 docs)     │     │  (Dynamic)      │     │  (OpenAI)       │
└─────────────────┘     └─────────────────┘     └────────┬────────┘
                                                          │
                                                          ▼
                                                ┌─────────────────┐
                                                │   FAISS INDEX   │
                                                │  (Vector Store) │
                                                └────────┬────────┘
                                                          │
┌─────────────────────────────────────────────────────────┼─────────────────┐
│                      QUERY PROCESSING                   │                 │
│                                                         │                 │
│  ┌─────────────────┐                          ┌─────────▼────────┐        │
│  │   USER QUERY    │─────────────────────────▶│  QUERY ANALYSIS  │        │
│  └─────────────────┘                          └─────────┬────────┘        │
│                                                         │                 │
│                              ┌──────────────────────────┴───────┐         │
│                              ▼                                  ▼         │
│                    ┌─────────────────┐              ┌─────────────────┐   │
│                    │  SINGLE-HOP     │              │   MULTI-HOP     │   │
│                    │  Multi-Query    │              │  Decomposition  │   │
│                    │  Expansion      │              │  + Sub-queries  │   │
│                    └────────┬────────┘              └────────┬────────┘   │
│                              │                                │           │
│                              └──────────────┬─────────────────┘           │
│                                             ▼                             │
│                               ┌─────────────────────┐                     │
│                               │   HYBRID RETRIEVAL  │                     │
│                               │  + MMR Diversity    │                     │
│                               └──────────┬──────────┘                     │
│                                          ▼                                │
│                               ┌─────────────────────┐                     │
│                               │ CONTEXT COMPRESSION │                     │
│                               │ + Source Attribution │                    │
│                               └──────────┬──────────┘                     │
│                                          ▼                                │
│                               ┌─────────────────────┐                     │
│                               │   LLM GENERATION    │                     │
│                               │ + Guardrail Prompt  │                     │
│                               └──────────┬──────────┘                     │
│                                          ▼                                │
│                               ┌─────────────────────┐                     │
│                               │  ANSWER VERIFICATION│                     │
│                               │ + Confidence Score  │                     │
│                               └──────────┬──────────┘                     │
└──────────────────────────────────────────┼────────────────────────────────┘
                                           ▼
                               ┌─────────────────────┐
                               │      RESPONSE       │
                               │  Answer + Sources   │
                               │  + Confidence       │
                               │  + Reasoning Trace  │
                               └─────────────────────┘
```

---
## Save Vector Store


In [ ]:
faiss_index.save("faiss_index")
print("✅ Index saved")

Saved index to faiss_index
✅ Index saved
